<a href="https://colab.research.google.com/github/SyedaMalaika75/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedaMalaika75/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Table used: fact_content_daily_performance.

Unit of analysis: One row represents one pseudonymized content item for one pseudonymized client on one report date.

I will use 1 March 2026 to 31 March 2026 as the development feature window. April 2026 will be used only for the later outcome proxy, while June 2026 will remain sealed as the final test month.

In [8]:
%pip -q install -U duckdb

from google.colab import userdata
import duckdb

# HF_TOKEN ko Colab Secrets se securely read karega
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN secret nahi mila. Colab Secrets mein HF_TOKEN add karke access ON karein."
    )

# DuckDB connection
con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# Token notebook output mein show nahi hoga
safe_token = hf_token.replace("'", "''")

con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{safe_token}'
)
""")

# Sirf March 2026 partition read hoga
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet',
    hive_partitioning = true
)
"""

# Query 1: grain verify karein
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_count
FROM {REL}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

# Query 2: row count aur date window verify karein
window_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(DISTINCT client_hash_id) AS client_count,
    COUNT(DISTINCT content_hash_id) AS content_count
FROM {REL}
""").df()

print("Query 1 — Duplicate grain rows:")
display(grain_check)

print("Query 2 — March 2026 row count and date span:")
display(window_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — Duplicate grain rows:


,report_date,client_hash_id,content_hash_id,duplicate_count


Query 2 — March 2026 row count and date span:


,row_count,start_date,end_date,client_count,content_count
0,9841378,2026-03-01,2026-03-31,55,331437


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

These are candidate features because they describe page performance available before the future ranking decision.

Label / proxy:
- Future search-visibility improvement for each content item.
- The proxy will be derived from later changes in impressions, clicks, and average search position.
- Label-period values will never be used as input features.

Context fields:
- report_date
- client_hash_id
- content_hash_id
- gsc_data_available
- ga4_data_available

The date and pseudonymized IDs will only be used for filtering, grouping, joining, and time-based validation. Availability flags will be used to determine whether the related metrics are genuinely usable.

Excluded fields:
- Client names, URLs, domains, raw search queries, and other identifying information.
- client_hash_id and content_hash_id as model features.
- Any future or label-derived fields.
- GA4 values where ga4_data_available IS NOT TRUE.
- GSC values where gsc_data_available IS NOT TRUE.
- June 2026 data during feature development because it is reserved as the final test or outcome month.

Human output:
A ranked, decision-support list of content items showing which pages should be reviewed or improved first.

In [9]:
field_contract = {
    "features": [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
    ],
    "label_or_proxy": [
        "future_search_visibility_improvement"
    ],
    "context": [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_data_available",
        "ga4_data_available",
    ],
    "excluded": [
        "identifying_information",
        "ids_as_model_features",
        "future_or_label_derived_fields",
        "unavailable_gsc_or_ga4_metrics",
        "june_2026_development_data",
    ],
}

for bucket, fields in field_contract.items():
    print(f"\n{bucket.upper()}:")
    for field in fields:
        print("-", field)



FEATURES:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

LABEL_OR_PROXY:
- future_search_visibility_improvement

CONTEXT:
- report_date
- client_hash_id
- content_hash_id
- gsc_data_available
- ga4_data_available

EXCLUDED:
- identifying_information
- ids_as_model_features
- future_or_label_derived_fields
- unavailable_gsc_or_ga4_metrics
- june_2026_development_data


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# Query 3: check how many March rows have both GSC and GA4 data available

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS rows_surviving,
    COUNT(DISTINCT client_hash_id) AS clients_surviving,
    COUNT(DISTINCT content_hash_id) AS content_items_surviving
FROM {REL}
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
""").df()

print("Query 3 — Rows with both GSC and GA4 data available:")
display(availability_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3 — Rows with both GSC and GA4 data available:


,rows_surviving,clients_surviving,content_items_surviving
0,364347,34,63856


Five-feature frame:

The five features use only measurements observed during March 2026. April 2026 is used only to create the later outcome proxy, so the feature and outcome windows do not overlap.

Available when?

1. gsc_impressions — available at the decision moment because it is calculated only from March GSC observations.

2. gsc_clicks — available at the decision moment because it is calculated only from March GSC observations.

3. gsc_avg_position — available at the decision moment because it uses only the page's observed March search position.

4. ga4_sessions — available at the decision moment because it uses only March GA4 sessions where GA4 data is genuinely available.

5. ga4_engaged_sessions — available at the decision moment because it uses only March engaged-session observations.

The outcome proxy is the change from average March daily impressions to average April daily impressions. It is used only as a label and never as a model feature.

In [11]:
# April 2026 data will only be used for the later outcome proxy.

REL_APRIL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet',
    hive_partitioning = true
)
"""

feature_frame = con.sql(f"""
WITH march_features AS (
    SELECT
        client_hash_id,
        content_hash_id,

        AVG(gsc_impressions) AS gsc_impressions,
        AVG(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        AVG(ga4_sessions) AS ga4_sessions,
        AVG(ga4_engaged_sessions) AS ga4_engaged_sessions,

        COUNT(DISTINCT report_date) AS march_days

    FROM {REL}

    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
),

april_outcome AS (
    SELECT
        client_hash_id,
        content_hash_id,

        AVG(gsc_impressions) AS april_daily_impressions,
        COUNT(DISTINCT report_date) AS april_days

    FROM {REL_APRIL}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,

    m.gsc_impressions,
    m.gsc_clicks,
    m.gsc_avg_position,
    m.ga4_sessions,
    m.ga4_engaged_sessions,

    a.april_daily_impressions - m.gsc_impressions
        AS future_impression_change

FROM march_features AS m

INNER JOIN april_outcome AS a
    USING (client_hash_id, content_hash_id)

WHERE m.march_days > 0
  AND a.april_days > 0
""").df()

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
]

print("Feature-frame shape:", feature_frame.shape)
print("Exactly five feature columns:", feature_columns)

display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame shape: (62681, 8)
Exactly five feature columns: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,future_impression_change
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,50.888889,0.222222,4.418032,1.555556,0.000000,-19.655556
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,157.720000,0.920000,4.392897,2.160000,0.040000,-37.186667
2,client_65de48885f4ef01b,content_3c286ded8bd68120,114.736842,0.789474,8.439390,1.578947,0.105263,-66.836842
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,35.928571,0.571429,5.531459,1.642857,0.071429,-23.395238
4,client_65de48885f4ef01b,content_ff867882e604fa96,12.000000,0.000000,2.850000,1.000000,0.000000,-7.500000


In [12]:
# Honest quick score:
# A valid March feature is compared with the later outcome proxy.

score_data = feature_frame[
    ["gsc_impressions", "future_impression_change"]
].dropna()

honest_score = (
    score_data["gsc_impressions"]
    .rank(method="average")
    .corr(
        score_data["future_impression_change"]
        .rank(method="average")
    )
)

# Deliberate leakage:
# This column directly copies the future label.

feature_frame["label_leak"] = feature_frame[
    "future_impression_change"
]

leak_data = feature_frame[
    ["label_leak", "future_impression_change"]
].dropna()

leaked_score = (
    leak_data["label_leak"]
    .rank(method="average")
    .corr(
        leak_data["future_impression_change"]
        .rank(method="average")
    )
)

print(f"Honest quick score: {honest_score:.4f}")
print(f"Leaked quick score: {leaked_score:.4f}")

# Remove the invalid leakage feature.

feature_frame.drop(
    columns=["label_leak"],
    inplace=True
)

print(
    "Leakage column removed:",
    "label_leak" not in feature_frame.columns
)

print("Final permitted feature columns:", feature_columns)

Honest quick score: -0.3249
Leaked quick score: 1.0000
Leakage column removed: True
Final permitted feature columns: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named limitation — availability and coverage bias:

Only records where both GSC and GA4 data are available can use all five proposed features. Therefore, the final usable dataset is a filtered subset of the complete March 2026 data and may overrepresent clients or content items with better analytics tracking.

Missing analytics data does not mean that the true metric value is zero. This dataset can show measured associations and support ranking decisions, but it cannot prove that a particular content change caused future search-performance improvement.

In [13]:
# Reuse the results of the three verification queries.
# This is not an additional warehouse query.

total_rows = int(window_check.iloc[0]["row_count"])
usable_rows = int(availability_check.iloc[0]["rows_surviving"])
usable_clients = int(availability_check.iloc[0]["clients_surviving"])
usable_items = int(availability_check.iloc[0]["content_items_surviving"])

print("Data-limit check — availability and coverage bias")
print(f"All March rows: {total_rows:,}")
print(f"Rows with both GSC and GA4 available: {usable_rows:,}")
print(f"Coverage rate: {usable_rows / total_rows:.2%}")
print(f"Clients surviving: {usable_clients:,}")
print(f"Content items surviving: {usable_items:,}")

print(
    "\nInterpretation: complete-feature analysis uses a filtered subset, "
    "so it may not represent every March 2026 record."
)


Data-limit check — availability and coverage bias
All March rows: 9,841,378
Rows with both GSC and GA4 available: 364,347
Coverage rate: 3.70%
Clients surviving: 34
Content items surviving: 63,856

Interpretation: complete-feature analysis uses a filtered subset, so it may not represent every March 2026 record.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.